In [1]:
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 8.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 97.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.7/184.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 15.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.1/888.1 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 MB 6.6 MB/s eta 0:00:00:00:0100:01
 

In [2]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-08-20 05:34:56.084724: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755668096.406900      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755668096.502233      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


Configuration Settings


In [3]:
max_seq_length = 1024
load_in_4bit = True

Loading the Phi-4 Model


In [4]:
import os
from huggingface_hub import login

# from google.colab import userdata
# huggingface_token = userdata.get('HF_TOKEN')
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
huggingface_token = user_secrets.get_secret("HF_TOKEN")

# Login to Hugging Face
if huggingface_token:
    login(huggingface_token)
else:
    raise ValueError("HF_TOKEN is not set in Colab secrets.")

In [5]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2025.8.8: Fast Llama patching. Transformers: 4.55.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Applying LoRA Adapters


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2025.8.8 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


* get_peft_model: A method to integrate LoRA adapters into the model for parameter-efficient fine-tuning.
* r=16: Sets the rank of the LoRA layers, controlling the dimensionality of the additional trainable parameters.
* target_modules: Specifies the model layers where LoRA adapters will be applied. These layers correspond to key components of the model’s transformer architecture.
* lora_alpha: A scaling factor for the LoRA layers to stabilize training.
* lora_dropout: Dropout probability for regularization; set to 0 for no dropout.
* bias=”none”: Indicates that no additional bias terms are introduced.
* use_gradient_checkpointing: Activates gradient checkpointing to reduce memory usage during backpropagation.
* random_state=3407: Ensures reproducibility by fixing the random seed.


###Step 2: Preparing the Dataset

We use the FineTome-100k dataset in ShareGPT format. The unsloth library provides utilities to convert this format into Hugging Face’s generic format for multi-turn conversations.

In [8]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt, get_chat_template

dataset = load_dataset("mlabonne/FineTome-100k", split="train")


README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [9]:
dataset = standardize_sharegpt(dataset)

Unsloth: Standardizing formats (num_proc=4):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [10]:
tokenizer = get_chat_template(tokenizer, chat_template="phi-4")

The get_chat_template function customizes the tokenizer to use the “phi-4” chat template. This ensures the prompts and conversations align with Phi-4’s format.

In [11]:
def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["conversations"]
    ]
    return {"text": texts}

The formatting_prompts_func processes each example in the dataset:

* The examples[“conversations”] field contains conversation data.
* Each conversation (convo) is passed through tokenizer.apply_chat_template.
* tokenize=False ensures the output is not tokenized yet.
* add_generation_prompt=False avoids appending generation-specific tokens to the prompts at this stage.
* The formatted text is stored under the “text” field.

Map Function to Dataset


In [12]:
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

The map function applies formatting_prompts_func to the entire dataset in batches. This efficiently preprocesses the dataset to prepare it for fine-tuning.

In [13]:
dataset[5]["conversations"]

[{'content': 'How do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?',
  'role': 'user'},
 {'content': 'Astronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.',
  'role': 'assistant'}]

###Step 3: Fine-Tuning the Model

Fine-tuning the Model involves training Phi-4 with Hugging Face’s SFTTrainer, optimizing the process with custom settings and efficient data handling.

Training with SFTTrainer

In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

* SFTTrainer: A specialized trainer for supervised fine-tuning of language models.
* TrainingArguments: Defines training hyperparameters such as batch size, learning rate, and number of steps.
* DataCollatorForSeq2Seq: Prepares input data for sequence-to-sequence models.
* is_bfloat16_supported: Checks if the system supports bfloat16, a mixed-precision format.

In [15]:
# from unsloth.chat_templates import train_on_responses_only

# trainer = train_on_responses_only(
#     trainer,
#     instruction_part="<|im_start|>user<|im_sep|>",
#     response_part="<|im_start|>assistant<|im_sep|>",
# )

In [16]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        max_steps=30,
        learning_rate=2e-4,
        # fp16=is_bfloat16_supported(),
        # bf16= not is_bfloat16_supported(), 
        fp16=True,
        bf16=False,  # T4s don’t support this
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        output_dir="outputs",
        report_to="none",
        # --- Added Checkpoint Settings ---
        save_strategy="steps",
        save_steps=10,            # Save every 10 steps
        save_total_limit=2,       # Optional: only keeps the last 2 checkpoints to save space
    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/100000 [00:00<?, ? examples/s]

####**Trainer Initialization:**

* model and tokenizer: The language model and its tokenizer are passed in, typically pre-configured.
* train_dataset: The dataset used for training, preprocessed and tokenized earlier.
* dataset_text_field: Specifies the field in the dataset containing the text.
* max_seq_length: The maximum sequence length for tokenized inputs.
* data_collator: Ensures input data is properly batched and padded.
* dataset_num_proc: Parallelizes dataset processing for efficiency.

####**Training Arguments:**

* per_device_train_batch_size: Batch size for each device during training (set to 2 here).
* gradient_accumulation_steps: Simulates a larger batch size by accumulating gradients over multiple steps.
* warmup_steps: Steps for learning rate warmup, helping stabilize training.
* max_steps: Total number of training steps (30 here, indicating a short training run).
* learning_rate: Learning rate for the optimizer.
* fp16 and bf16: Enable mixed precision (FP16 or BF16) based on hardware support for faster and memory-efficient training.
* logging_steps: Frequency of logging during training.
* optim: Optimizer choice; adamw_8bit reduces memory usage.
* weight_decay: Regularization parameter to prevent overfitting.
* output_dir: This directory saves the model checkpoints and logs.
* report_to: Disables reporting to external tracking tools (e.g., WandB).
Purpose:

**This setup efficiently fine-tunes a large model on a custom dataset, focusing on:**

* Memory optimization (e.g., mixed precision, 8-bit optimizers).
* Efficient training configurations with a small batch size and gradient accumulation.
* Short, lightweight training for quick experimentation or domain adaptation.

We can also use Unsloth’s train_on_completions method to only train on the assistant outputs and ignore the loss on the user’s inputs.

Masking User Inputs:

To train only on assistant responses, we mask user inputs using the train_on_responses_only utility:

In [17]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user<|im_sep|>",
    response_part="<|im_start|>assistant<|im_sep|>",
)

Map (num_proc=4):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [18]:
#Let’s verify masking is actually done:

tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|im_start|>user<|im_sep|>How do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?<|im_end|><|im_start|>assistant<|im_sep|>Astronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.<|im_end|>'

In [19]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                     Astronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.<|im_end|>'

###Step 4: Monitoring GPU Usage


Check GPU memory usage before and after training:



In [20]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
10.727 GB of memory reserved.


### Step 5: Inference

In [21]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
tokenizer,
chat_template = "phi-4",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(100352, 5120, padding_idx=100351)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=5120, out_features=5120, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=5120, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=5120, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear4b

In [22]:
messages = [
    {"role": "user", "content": "Continue the Fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]

Preprocessing Inputs with the Tokenizer


In [23]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

apply_chat_template: Prepares the input for the Phi-4 model using the tokenizer and ensures compatibility with the chat format.

Parameters:

* tokenize=True: Converts text into token IDs.
* add_generation_prompt=True: Adds a special prompt token to guide the model’s response generation.
* return_tensors=”pt”: Converts the processed data into PyTorch tensors for GPU processing.
* .to(“cuda”): Moves the data to the GPU for accelerated computation.


Generating Text:



In [24]:
outputs = model.generate(
    input_ids=inputs, max_new_tokens=64, use_cache=True, temperature=1.5, min_p=0.1
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Parameters:

* input_ids=inputs: The tokenized input.
* max_new_tokens=64: Limits the length of the generated output to 64 tokens.
* use_cache=True: Speeds up generation by using cached activations.
* temperature=1.5: Controls randomness in output (higher values = more creative, less deterministic).
* min_p=0.1: Ensures diversity by setting a minimum probability threshold for token sampling.

Decoding and Displaying the Output:

In [25]:
print(tokenizer.batch_decode(outputs))

['<|im_start|>user<|im_sep|>Continue the Fibonacci sequence: 1, 1, 2, 3, 5, 8,<|im_end|><|im_start|>assistant<|im_sep|>To continue the Fibonacci sequence, each number is the sum of the two preceding ones. Starting from the last two numbers given, 5 and 8:\n\n- \\(5 + 8 = 13\\)\n- \\(8 + 13 = 21\\)\n- \\(13 + 21 = 34\\)\n']


Decodes the generated token IDs back into human-readable text using the tokenizer.

We use batch_decode because the outputs might contain multiple sequences.

In [26]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
{"role": "user", "content": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
messages,
tokenize = True,
add_generation_prompt = True, # Must add for generation
return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(
input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
use_cache = True, temperature = 1.5, min_p = 0.1
)

To continue the Fibonacci sequence, each number is the sum of the two preceding ones. Starting from the sequence you provided:

1, 1, 2, 3, 5, 8

The next number is:

8 + 5 = 13

Continuing further:

13 + 8 = 21

21 + 13 = 34

34 + 21 = 55

So, the sequence continues as:

1, 1, 2, 3, 5, 8, 13, 21, 34, 55<|im_end|>


In [27]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/vocab.json',
 'lora_model/merges.txt',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

### Saving the LoRA adapter and tokenizer to disk, then pushing that folder using Hugging Face tools.

In [33]:
# from google.colab import userdata
# hf_token = userdata.get('HF_TOKEN')

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

output_dir = "phi-4-finetuned"

# Save LoRA adapter
model.save_pretrained(output_dir, safe_serialization=True)

# Save tokenizer
tokenizer.save_pretrained(output_dir)

('phi-4-finetuned/tokenizer_config.json',
 'phi-4-finetuned/special_tokens_map.json',
 'phi-4-finetuned/chat_template.jinja',
 'phi-4-finetuned/vocab.json',
 'phi-4-finetuned/merges.txt',
 'phi-4-finetuned/added_tokens.json',
 'phi-4-finetuned/tokenizer.json')

In [35]:
# 3. Upload folder contents
from huggingface_hub import upload_folder

upload_folder(
    repo_id="christina444/phi-4-finetuned",
    folder_path=output_dir,
    path_in_repo=".",
    token=hf_token
)

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/131M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/christina444/phi-4-finetuned/commit/1cda2a62e25dbc4339f5a3df41036afb229dfdd1', commit_message='Upload folder using huggingface_hub', commit_description='', oid='1cda2a62e25dbc4339f5a3df41036afb229dfdd1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/christina444/phi-4-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='christina444/phi-4-finetuned'), pr_revision=None, pr_num=None)

#### LoRA fine-tuned model and tokenizer were successfully uploaded to Hugging Face!

In [ ]:
##To Load https://huggingface.co/christina444/phi-4-finetuned 

# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import PeftModel

# # Load base + LoRA adapter
# base_model = AutoModelForCausalLM.from_pretrained("unsloth/phi-4", device_map="auto")
# model = PeftModel.from_pretrained(base_model, "christina444/phi-4-finetuned")
# tokenizer = AutoTokenizer.from_pretrained("christina444/phi-4-finetuned")
